# Train A Medical Model From Scratch

In this notebook, we take a general Qwen model and teach it how to answer biomedical research questions using PubMedQA medical data.

The journey is simple: give Gradients a dataset, let the network train the model, then test it on medical questions it has never seen before. At the end, you will see the base model and trained model side by side, with the expected answer from the test set.

No training scripts, no infrastructure work, no ML ops setup. Just an API key, a dataset, and a few notebook cells.

## Step 1: Install The Gradients SDK

One install gives this notebook everything it needs to launch training, sample data, load models, merge adapters, and run inference.

In [ ]:
%pip install -q --upgrade gradientsio==0.1.2

## Step 2: Choose The Mission

We will start with `Qwen/Qwen2.5-3B`, train it for 2 hours on normalized PubMedQA examples, and keep a separate PubMedQA test set untouched for the final showdown.

Paste your Gradients API key when asked. Everything else is already filled in.

In [ ]:
import os
import time
from getpass import getpass

from IPython.display import Markdown
from IPython.display import display
from gradientsio import GenerationConfig
from gradientsio import GradientsClient
from gradientsio import ModelSampler
from gradientsio import TaskType
from gradientsio import load_dataset_rows

MODEL_ID = "Qwen/Qwen2.5-3B"
TRAIN_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Train"
TEST_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Test"
HOURS_TO_TRAIN = 2
RESULT_MODEL_NAME = "pubmedqa-qwen2-5-3b-gradients-demo"
SAMPLE_SIZE = 5
SAMPLE_SEED = 23223
POLL_INTERVAL_SECONDS = 300
REQUIRE_CUDA = True
GENERATION = GenerationConfig(max_new_tokens=96, repetition_penalty=1.12, num_beams=4)

if not os.getenv("GRADIENTS_API_KEY"):
    os.environ["GRADIENTS_API_KEY"] = getpass("Gradients API key: ").strip()

client = GradientsClient()

## Step 3: Send The Model To Training

This is the zero-faff part: we point Gradients at the medical training dataset, pick the base model, and launch the job.

The printed task ID is your receipt. If you close the notebook, paste that ID into the next step and continue where you left off.

In [ ]:
task = client.train(
    model=MODEL_ID,
    task_type=TaskType.INSTRUCT,
    hours=HOURS_TO_TRAIN,
    dataset=TRAIN_DATASET,
    field_instruction="instruction",
    field_input="input",
    field_output="output",
    result_model_name=RESULT_MODEL_NAME,
)

TASK_ID = task.task_id
print(f"Training task created: {TASK_ID}")

## Step 4: Wait For The Trained Model

Gradients now does the heavy lifting: dataset prep, scheduling, training, evaluation, and publishing the trained model repo.

Run this cell after launch. If you already have a task ID from an earlier run, paste it in and the notebook will wait for the final trained model.

In [ ]:
if "TASK_ID" not in globals() or not TASK_ID:
    TASK_ID = input("Paste an existing Gradients task ID: ").strip()

details = client.tasks.handle(TASK_ID).wait(
    poll_interval=POLL_INTERVAL_SECONDS,
    raise_on_failure=True,
)

TRAINED_MODEL_REPO = details.trained_model_repository
if not TRAINED_MODEL_REPO:
    raise RuntimeError("Training succeeded, but no trained_model_repository was returned.")

print(f"Training complete: {TRAINED_MODEL_REPO}")

## Step 5: Pick Questions The Model Has Never Seen

Now we pull a few examples from the held-out test set. These are not part of the training run.

This is where the story gets interesting: can a small base model learn the medical answer style from your custom data, then apply it to fresh biomedical questions?

In [ ]:
samples = load_dataset_rows(TEST_DATASET, sample_size=SAMPLE_SIZE, seed=SAMPLE_SEED)


def build_prompt(row):
    instruction = (row.get("instruction") or "").strip()
    return f"{instruction}\n\nAnswer:"


for index, row in enumerate(samples, start=1):
    prompt = build_prompt(row)

## Step 6: Watch The Before And After

Time for the proof.

We ask the original base model and the newly trained model the same unseen medical questions. Then we place both answers next to the expected PubMedQA answer so the improvement is easy to judge at a glance.

This is the whole Gradients loop: bring your data, train your model, test the difference.

In [ ]:
if "TRAINED_MODEL_REPO" not in globals() or not TRAINED_MODEL_REPO:
    TRAINED_MODEL_REPO = input("Paste the trained_model_repository from Gradients: ").strip()

samples = load_dataset_rows(TEST_DATASET, sample_size=SAMPLE_SIZE, seed=SAMPLE_SEED)
prompts = [build_prompt(row) for row in samples]

sampler = ModelSampler(require_cuda=REQUIRE_CUDA)

base_answers = sampler.generate(MODEL_ID, prompts, config=GENERATION)
trained_answers = sampler.generate_with_adapter(
    TRAINED_MODEL_REPO,
    prompts,
    base_model_repo=MODEL_ID,
    config=GENERATION,
)

sections = []
for index, row in enumerate(samples, start=1):
    question = prompts[index - 1].removesuffix("Answer:").strip()
    expected = (row.get("output") or "").strip()
    trained = trained_answers[index - 1].strip()
    base = base_answers[index - 1].strip()
    sections.append(
        f"""
---

## Example {index} | PubMed ID: {row.get('pubid')}

### Question

{question}

### Expected Answer

{expected}

### Trained Model Answer

{trained}

### Base Model Answer

{base}
"""
    )

display(Markdown("\n".join(sections)))